# Build CNN Classification Dataset from YOLO Labels

Crops each card from the synthetic images using ground-truth YOLO bounding boxes,
adds configurable padding, and **resizes (warps) to a fixed square** (default 227×227)
to match what the CNN will see at inference time when fed raw YOLO detection crops.

Output: `cnn_dataset/<split>/<class_name>/*.jpg` — ready for `torchvision.datasets.ImageFolder`.

In [ ]:
# ── Class names (must match data.yaml index order) ──────────
card_names = [
    '10c', '10d', '10h', '10s', '2c', '2d', '2h', '2s',
    '3c', '3d', '3h', '3s', '4c', '4d', '4h', '4s',
    '5c', '5d', '5h', '5s', '6c', '6d', '6h', '6s',
    '7c', '7d', '7h', '7s', '8c', '8d', '8h', '8s',
    '9c', '9d', '9h', '9s', 'Ac', 'Ad', 'Ah', 'As',
    'Jc', 'Jd', 'Jh', 'Js', 'Kc', 'Kd', 'Kh', 'Ks',
    'Qc', 'Qd', 'Qh', 'Qs',
]

In [ ]:
def yolo_to_pixels(box_line, img_width, img_height):
    """Convert a YOLO-format label line to pixel coords + class name."""
    parts = box_line.strip().split()
    class_id = int(parts[0])
    x_center = float(parts[1]) * img_width
    y_center = float(parts[2]) * img_height
    w = float(parts[3]) * img_width
    h = float(parts[4]) * img_height

    x1 = max(0, x_center - w / 2)
    y1 = max(0, y_center - h / 2)
    x2 = min(img_width,  x_center + w / 2)
    y2 = min(img_height, y_center + h / 2)

    card_name = card_names[class_id]
    return x1, y1, x2, y2, card_name

In [ ]:
from PIL import Image
from tqdm import tqdm
import os

# ── Configuration ────────────────────────────────────────────
CNN_INPUT_SIZE = 227        # Target square size (e.g. 227 for AlexNet)
PAD_FRACTION   = 0.25      # Fractional padding added to each side of the bbox

source_dirs = {
    'train': './train/images',
    'valid': './valid/images',
    'test':  './test/images',
}
label_dirs = {
    'train': './train/labels',
    'valid': './valid/labels',
    'test':  './test/labels',
}
base_dir = './cnn_dataset'

# ── Create output folders ────────────────────────────────────
for split in ['train', 'valid', 'test']:
    for card in card_names:
        os.makedirs(os.path.join(base_dir, split, card), exist_ok=True)

# ── Extract, pad, warp-resize, and save ─────────────────────
for split in ['train', 'valid', 'test']:
    src_dir = source_dirs[split]
    lbl_dir = label_dirs[split]
    out_dir = os.path.join(base_dir, split)

    all_images = sorted([f for f in os.listdir(src_dir)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Processing {split} — {len(all_images)} images')

    for img_name in tqdm(all_images):
        img_path   = os.path.join(src_dir, img_name)
        stem       = os.path.splitext(img_name)[0]
        label_path = os.path.join(lbl_dir, stem + '.txt')

        if not os.path.exists(label_path):
            continue

        img = Image.open(img_path).convert('RGB')
        W, H = img.size

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            if not line.strip():
                continue

            x1, y1, x2, y2, card_name = yolo_to_pixels(line, W, H)

            # ── Add padding (proportional to bbox dimensions) ──
            bw = x2 - x1
            bh = y2 - y1
            pad_x = PAD_FRACTION * bw
            pad_y = PAD_FRACTION * bh

            cx1 = max(0, x1 - pad_x)
            cy1 = max(0, y1 - pad_y)
            cx2 = min(W, x2 + pad_x)
            cy2 = min(H, y2 + pad_y)

            # ── Crop and warp-resize to CNN_INPUT_SIZE × CNN_INPUT_SIZE ──
            # This is an intentional non-aspect-preserving resize (warp).
            # It matches what happens at inference when we crop a YOLO
            # detection (arbitrary aspect ratio) and feed it to the CNN.
            crop = img.crop((cx1, cy1, cx2, cy2))
            crop = crop.resize((CNN_INPUT_SIZE, CNN_INPUT_SIZE), Image.BILINEAR)

            save_path = os.path.join(out_dir, card_name, f'{stem}_{i}.jpg')
            crop.save(save_path, quality=95)

print('\nDone.')

### Quick Sanity Check

Preview some crops and verify they look like what the CNN will see at inference.

In [ ]:
import random
import matplotlib.pyplot as plt
import pathlib

crop_root = pathlib.Path(base_dir) / 'train'
all_crops = list(crop_root.rglob('*.jpg'))
samples   = random.sample(all_crops, min(16, len(all_crops)))

fig, axes = plt.subplots(2, 8, figsize=(22, 6))
for ax, fp in zip(axes.flat, samples):
    img = Image.open(fp)
    ax.imshow(img)
    ax.set_title(f'{fp.parent.name}\n{img.size[0]}×{img.size[1]}', fontsize=8)
    ax.axis('off')
plt.suptitle(f'Random CNN crops — warped to {CNN_INPUT_SIZE}×{CNN_INPUT_SIZE}', fontsize=13)
plt.tight_layout()
plt.show()

# ── Per-class counts ──────────────────────────────────────────
from collections import Counter
counts = Counter(fp.parent.name for fp in crop_root.rglob('*.jpg'))
total  = sum(counts.values())
print(f'Total training crops: {total:,}')
print(f'Classes with crops:   {len(counts)}/{len(card_names)}')
print(f'Min per class: {min(counts.values()):,}  |  Max: {max(counts.values()):,}')